### Determine suitable thresholds for the baseline method

In [ ]:
# imports

# external modules
import os
import sys
import json
import numpy as np
import scipy as sp
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from fnmatch import fnmatch
import importlib

# local modules
thisdir = os.getcwd()
topdir = os.path.abspath(os.path.join(thisdir, '../../../'))
sys.path.append(topdir)

import tools.iotools as iotools
import tools.dftools as dftools
import tools.patternfiltering as patternfiltering
from tools.dataloadertools import MEDataLoader
import studies.ecal_occupancy_2024.preprocessing.preprocessor
importlib.reload(studies.ecal_occupancy_2024.preprocessing.preprocessor)
from studies.ecal_occupancy_2024.preprocessing.preprocessor import make_default_preprocessor
from studies.ecal_occupancy_2024.plotting.plot_cluster_occupancy import plot_cluster_occupancy

In [ ]:
# settings

# settings
datadir = '/eos/user/l/llambrec/dialstools-output-test'
year = '2024'
eras = [
    'B-v1',
    'C-v1',
    'D-v1',
    'E-v1',
    'E-v2',
    'F-v1',
    'G-v1',
    'H-v1',
    'I-v1',
    'I-v2'
]
dataset = 'ZeroBias'
reco = 'PromptReco'
menames = {
    'EB': 'EcalBarrel-EBOccupancyTask-EBOT digi occupancy',
    #'EE+': 'EcalEndcap-EEOccupancyTask-EEOT digi occupancy EE +',
    #'EE-': 'EcalEndcap-EEOccupancyTask-EEOT digi occupancy EE -'
}

# find files corresponding to settings
files = {}
for era in eras:
    files[era] = {}
    for melabel, mename in menames.items():
        mainera, version = era.split('-', 1)
        f = f'{dataset}-Run{year}{mainera}-{reco}-{version}-DQMIO-{mename}.parquet'
        f = os.path.join(datadir, f)
        files[era][melabel] = f

# existence check
missing = []
for era, temp in files.items():
    for melabel, f in temp.items():
        if not os.path.exists(f):
            missing.append(f)
if len(missing) > 0:
    raise Exception(f'The following files do not exist: {missing}')

In [ ]:
# set path to OMS files

omsfiles = {}
for era in eras:
    basedir = '../omsdata'
    omsfile = os.path.join(basedir, f'omsdata_Run{year}{era}.json')
    if not os.path.exists(omsfile):
        print(f'WARNING: OMS file {omsfile} does not exist.')
    omsfiles[era] = omsfile

In [ ]:
# set path to HLT rate files
# (just use the ones from pixel study as we're dealing with the same ZeroBias triggers)

hltfiles = {}
for era in eras:
    basedir = '../../pixel_clusters_2024/omsdata'
    hltfile = os.path.join(basedir, f'hltrate_Run{year}{era}.json')
    if not os.path.exists(hltfile):
        print(f'WARNING: HLT rate file {hltfile} does not exist.')
    hltfiles[era] = hltfile

In [ ]:
# define preprocessing era mapping

#preprocessing_era_map = {
#    'B-v1': 'C-v1',
#    'C-v1': 'B-v1',
#    'D-v1': 'C-v1',
#    'E-v1': 'D-v1',
#    'E-v2': 'E-v1',
#    'F-v1': 'E-v2',
#    'G-v1': 'F-v1',
#    'H-v1': 'G-v1',
#    'I-v1': 'H-v1',
#    'I-v2': 'I-v1'
#}

preprocessing_era_map = {era: 'C-v1' for era in eras}

In [ ]:
from studies.pixel_clusters_2024.nmf.nmf_testing_pattern import filter_dfs # make local equivalent later

# loop over mes
quantiles = {}
for melabel, mename in menames.items():
    
    # loop over eras
    quantiles[melabel] = {}
    for era in eras:
        print(f'Now processing me {mename}, era {era}...')
        
        # make dataloader
        dataloader = MEDataLoader([files[era][melabel]])
        preprocessor_era = preprocessing_era_map[era]
        preprocessor = make_default_preprocessor(preprocessor_era, melabel,
                     global_normalization = 'avg',
                     local_normalization = 'avg')
        
        # make loss mask
        loss_mask_era = preprocessing_era_map[era]
        loss_mask_file = f'../preprocessing/normdata/zerofrac_Run2024{loss_mask_era}_{melabel}.npy'
        loss_mask = np.load(loss_mask_file)
        loss_mask = (loss_mask < 0.9).astype(bool)
        
        # load OMS data
        with open(omsfiles[era]) as f:
            oms_info = json.load(f)
            
        # load HLT rate data
        with open(hltfiles[era]) as f:
            hltrate_info = json.load(f)
            
        # initializations
        batch_quantiles = []
        batch_highnbins = []
        
        # run over batches
        batch_size = 20000
        batch_params = dataloader.prepare_sequential_batches(batch_size=batch_size)
        print(f'Will process {len(batch_params)} batches of size {batch_size}')
        for batch_idx, batch_param in enumerate(batch_params):
            batch = dataloader.read_sequential_batch(batch_param)
            batch_size = len(batch)
            print(f'  Loaded batch {batch_idx+1} / {len(batch_params)} with size {batch_size}')
            
            # filtering
            fdict = {melabel: batch}
            oms_filters = [
                ['beams_stable'],
                ['cms_active'],
                ['ebm_ready'],
                ['ebp_ready'],
                ['eem_ready'],
                ['eep_ready'],
                ['esm_ready'],
                ['esp_ready'],
                ['pileup', '>', 10]
            ]
            hltrate_filters = [
                ["HLT_ZeroBias_v*", '>', 25]
            ]
            mask, _ = filter_dfs(fdict,
                                     oms_info = oms_info,
                                     oms_filters = oms_filters,
                                     hltrate_info = hltrate_info,
                                     hltrate_filters = hltrate_filters
                                    )
            mask = mask.astype(bool)
            batch = batch[mask]
            batch_size_filtered = len(batch)
            print(f'  Found {batch_size_filtered} / {batch_size} instances passing filters.')
            if batch_size_filtered==0: continue
                
            # update, hard-coded for now, to be generalized later:
            # get a more fine-grained local normalization.
            # note: just a rough first version, taking the proper normalization for the first run in the batch
            #       and assume it is valid for the whole batch; this is fine in most cases on the condition
            #       that the evaluation batch size is the same or much smaller than the batches used to make the normalization.
            first_run = int(batch['run_number'].values[0])
            candidates = sorted([f for f in os.listdir(f'../preprocessing/normdata/') if fnmatch(f, f'avgme_*_{melabel}_*')])
            norm_idx = -1
            for candidate_idx, candidate in enumerate(candidates):
                last_run = int(candidate.split('run')[-1].split('-')[-1].replace('.npy', ''))
                if last_run >= first_run:
                    norm_idx = candidate_idx - 1
                    break
            if norm_idx < 0: norm_idx = 0
            normfile = candidates[norm_idx]
            normfile = os.path.join('../preprocessing/normdata', normfile)
            normmap = np.load(normfile)
            normmap[normmap==0] = -1
            preprocessor.local_norm = normmap
            
            # preprocessing
            mes_preprocessed = preprocessor.preprocess(batch)
            
            # set masked out bins to nan
            temp_mask = np.broadcast_to(loss_mask, mes_preprocessed.shape)
            mes_preprocessed[~temp_mask] = np.nan
            
            # collect quantiles per ME
            batch_quantiles_result = {}
            qs = [0.01, 0.1, 0.5, 0.9, 0.99]
            for q in qs:
                batch_quantiles_result[str(q)] = list(np.nanquantile(mes_preprocessed, q, axis=(1,2)))
                batch_quantiles_result['max'] = list(np.nanmax(mes_preprocessed, axis=(1,2)))
                batch_quantiles_result['min'] = list(np.nanmin(mes_preprocessed, axis=(1,2)))
            batch_quantiles.append(batch_quantiles_result)
                
        # contatenate the results from the batches
        this_quantiles = {}
        if len(batch_quantiles)>0:
            for key in batch_quantiles[0].keys():
                this_quantiles[key] = np.array(sum([el[key] for el in batch_quantiles], []))
                
        # add to the result
        quantiles[melabel][era] = this_quantiles

In [ ]:
# make a plot of the quantiles

# settings
plot_melabel = 'EB'
plot_eras = ['all']
#plot_eras= ['D-v1', 'I-v2']

# fixed settings
colors = {
    'min': 'blue',
    '0.01': 'dodgerblue',
    '0.1': 'lightskyblue',
    '0.5': 'grey',
    '0.9': 'orchid',
    '0.99': 'crimson',
    'max': 'red'
}
if 'all' in plot_eras: plot_eras = list(quantiles[plot_melabel].keys())

# make x-axis
xax = {}
counter = 0
for era in plot_eras:
    thislen = len(quantiles[plot_melabel][era]['min'])
    xax[era] = np.arange(counter, counter+thislen)
    counter = counter + thislen
    
# make the base figure
fig, ax = plt.subplots(figsize=(18,6))
for era_idx, era in enumerate(plot_eras):
    this_xax = xax[era]
    this_quantiles = quantiles[plot_melabel][era]
    minlabel = 'min.' if era_idx==0 else None
    ax.plot(this_xax, this_quantiles['min'], label=minlabel, color=colors['min'])
    for key, q in this_quantiles.items():
        if key in ['min', 'max']: continue
        label = key + ' quantile' if era_idx==0 else None
        ax.plot(this_xax, q, label=label, color=colors[key])
    maxlabel = 'max.' if era_idx==0 else None
    ax.plot(this_xax, this_quantiles['max'], label=maxlabel, color=colors['max'])
    
# plot aesthetics
ax.legend(fontsize=15, loc='upper left', bbox_to_anchor=(1.01, 0, 0.2, 1))
ax.set_yscale('log')
ax.set_xlabel('Time (lumisections)', fontsize=15)
ax.set_ylabel('Value', fontsize=15)
ax.set_ylim((0.1, 10))
ax.grid(which='both', axis='y')

# add era separators
for era in plot_eras:
    idx = xax[era][0]
    ax.axvline(idx, color='grey', linestyle='dashed')
    txt = ax.text(idx+300, 9, era, ha='left', va='top', fontsize=15)
    txt.set_bbox(dict(facecolor='white', alpha=0.5))